# 📧 Llama 3.1 8B Instruct — QLoRA Fine-Tuning on Enron Emails

**Clean and modular training pipeline using custom repository modules.**

This notebook walks through:
1. Environment setup & Cloning the project repository
2. Data preparation (using the raw maildir and CLI tools)
3. QLoRA fine-tuning using `training.train_lora`
4. Custom email metrics evaluation using `evaluation.metrics`
5. Quick inference testing with the trained adapter

> ⚠️ **Requires**: HuggingFace account with Llama 3.1 access + HF token

---
## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install core training and evaluation dependencies
!pip install -q \
    torch \
    transformers>=4.43.0 \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    trl>=0.9.0 \
    datasets>=2.20.0 \
    accelerate>=0.33.0 \
    rouge-score \
    nltk \
    evaluate \
    bert-score \
    tensorboard \
    scikit-learn

In [ ]:
# Clone the project repository and set as the working directory
# TODO: Replace the URL below with your actual fork or repository URL
import os
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME.git"
REPO_NAME = "fine_tune"

if not os.path.exists(REPO_NAME) and not os.path.exists(".git"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}

In [ ]:
# Login to HuggingFace (required for gated Llama 3.1 model access)
from huggingface_hub import login
login()

---
## 2. Data Preparation

You can run the data preparation pipeline script directly inside Google Colab using the cloned repository scripts, or upload your preprocessed JSONL files (`train.json` and `val.json`).

In [ ]:
# Option A: Run preparation script on raw maildir folder
# First upload the raw 'maildir' folder to Colab (e.g., in '/content/maildir') or download it
# !python data/prepare_data.py --maildir_path /content/maildir --max_samples 10000 --output_dir data/

In [ ]:
# Option B: Upload train.json and val.json to your 'data' folder using the files panel

In [ ]:
# Verify data files
import json
TRAIN_FILE = "data/train.json"
VAL_FILE = "data/val.json"

def count_samples(filepath):
    count = 0
    task_counts = {}
    with open(filepath, 'r') as f:
        for line in f:
            data = json.loads(line.strip())
            count += 1
            task = data.get('task_type', 'unknown')
            task_counts[task] = task_counts.get(task, 0) + 1
    return count, task_counts

try:
    train_count, train_tasks = count_samples(TRAIN_FILE)
    val_count, val_tasks = count_samples(VAL_FILE)
    print(f"Training samples: {train_count} {train_tasks}")
    print(f"Validation samples: {val_count} {val_tasks}")
except FileNotFoundError:
    print("Please make sure train.json and val.json are placed in the data/ directory.")

---
## 3. QLoRA Fine-Tuning

We import our centralized training configurations and run the end-to-end training module.

In [ ]:
from training.config import get_config
from training.train_lora import train

# 1. Load pipeline configuration
config = get_config()

# 2. Customize parameters if needed (batch size, epochs, paths etc.)
config.training.num_train_epochs = 3
config.training.learning_rate = 2e-4
config.data.train_file = "data/train.json"
config.data.val_file = "data/val.json"

config.training.batch_size = 1
config.training.gradient_accumulation_steps = 2
config.training.max_seq_length = 128

print(config.summary())

# 3. Run QLoRA training
train(config)

---
## 4. TensorBoard Dashboard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./logs

---
## 5. Model Evaluation using MetricCalculator

In [ ]:
from evaluation.metrics import MetricCalculator

# Initialize MetricCalculator
calculator = MetricCalculator()

# Example prediction/reference test comparison
predictions = ["Dear Team,\n\nPlease review the attached document containing the budget calculations.\n\nRegards,\nManagement"]
references = ["Dear Team,\n\nPlease review the attached budget overview files ahead of our meeting.\n\nThanks,\nManagement"]

# Compute Rouge, Bleu, Format compliance and Length stats (skipping BERTScore by default to save time/ram)
results = calculator.compute_all(predictions, references, skip_bertscore=True)
print(json.dumps(results, indent=2))

---
## 6. Inference Test

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

ADAPTER_PATH = "./outputs/final_adapter"

# Setup quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # T4 compute capability
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

print("Loading adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Model ready for generation!")

In [ ]:
# Run composition generation test
test_messages = [
    {"role": "system", "content": "You are a professional email writing assistant. Write clear, concise, and professional emails based on the given subject and context."},
    {"role": "user", "content": "Write a professional email with the subject: Project Scope and Deliverables Delay"}
]

input_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"\nGenerated Output:\n{'-'*50}\n{response}")

---
## 7. Save and Hub Export (Optional)

In [ ]:
# Copy adapter back to mounted Google Drive folder for persistent storage
# import shutil
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree("./outputs/final_adapter", "/content/drive/MyDrive/fine_tune/outputs/final_adapter", dirs_exist_ok=True)

In [ ]:
# Push adapter to Hugging Face Hub
# HUB_MODEL_ID = "YOUR_HF_USERNAME/llama-3.1-8b-enron-qlora"
# model.push_to_hub(HUB_MODEL_ID)
# tokenizer.push_to_hub(HUB_MODEL_ID)